# LoRA
"LoRA: Low-Rank Adaptation of Large Language Models" (Hu et al., 2021)

**Idea:** to fine-tune a pretrained model, **freeze** every weight matrix $W$ and learn only a small **low-rank update** next to it: $W + BA$, where $B$ and $A$ are thin matrices of rank $r$. The hypothesis is that the change needed to adapt a model to a new task lives in a low-dimensional subspace, so a rank of 4-16 is enough even when $W$ is thousands of dimensions wide. After training, $BA$ can be added into $W$, so inference costs nothing extra.

### Ways to adapt a pretrained model
| | Full fine-tuning | Last layer / head only | Adapters (Houlsby et al.) | Prompt / prefix tuning | LoRA |
|---|---|---|---|---|---|
| Trainable params | 100% | < 1% | ~1-4% | < 0.1% | **~0.1-1%** |
| What is learned | all weights | output layer | small MLPs inserted between layers | virtual tokens prepended to the input | **low-rank $\Delta W$** next to chosen matrices |
| Extra inference cost | none | none | extra layers (latency) | shorter usable context | **none after merging** |
| Storage per task | a full model copy | tiny | small | tiny | **a few MB** |
| Quality | best / reference | limited | close to full | varies | **close to full** |
| Swap tasks at runtime | reload model | yes | yes | yes | yes (add / subtract $BA$) |

Here: pretrain a small character-level decoder on Shakespeare, then adapt it to a different distribution (first names) with full fine-tuning and with LoRA at several ranks; merge the weights; look at the rank of the learned update.

Shape flow of one adapted layer: `x (.., D_in) -> W x (frozen) + (alpha / r) * B (A x) -> (.., D_out)` with `A (r, D_in)`, `B (D_out, r)`

In [ ]:
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## 1. Two datasets
- **Pretraining**: tiny Shakespeare, character level (as in `slm.ipynb`).
- **Adaptation**: `names.txt`, 32k first names, one per line (the makemore data). Same characters, completely different statistics: short lowercase lines instead of verse.

Both use the Shakespeare character vocabulary, so the pretrained embedding table applies to the new task unchanged.

In [ ]:
text = open("../datasets/tiny_shakespeare.txt").read()
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

names = open("../datasets/names.txt").read()               # "emma\nolivia\nava\n..."
assert set(names) <= set(chars)

def split(s):
    d = torch.tensor(encode(s))
    n = int(0.9 * len(d))
    return {"train": d[:n], "val": d[n:]}

datasets = {"shakespeare": split(text), "names": split(names)}
block_size, batch_size = 64, 32   # T, B

def get_batch(task, split):
    # -> x (B, T), y (B, T); y is x shifted left by one
    d = datasets[task][split]
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

print({k: len(v["train"]) for k, v in datasets.items()}, "| vocab:", vocab_size)

## 2. The base model
A small Pre-LN decoder (see `slm.ipynb` for the block; learned positions and a GELU MLP here to keep it short). The query, key, value and output projections are **separate `nn.Linear` layers** so that LoRA can be attached to individual matrices, as in the paper.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.q, self.k, self.v, self.o = (nn.Linear(dim, dim, bias=False) for _ in range(4))   # W_q, W_k, W_v, W_o

    def forward(self, x):
        B, T, D = x.shape
        heads = lambda t: t.view(B, T, self.num_heads, D // self.num_heads).transpose(1, 2)   # (B, heads, T, hd)
        out = F.scaled_dot_product_attention(heads(self.q(x)), heads(self.k(x)), heads(self.v(x)), is_causal=True)
        return self.o(out.transpose(1, 2).reshape(B, T, D))


class Block(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.ln1, self.attn = nn.LayerNorm(dim), Attention(dim, num_heads)
        self.ln2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(), nn.Linear(4 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))


class GPT(nn.Module):
    # idx (B, T) -> logits (B, T, V)
    def __init__(self, vocab_size, block_size, dim=192, depth=4, num_heads=4):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab_size, dim), nn.Embedding(block_size, dim)
        self.blocks = nn.ModuleList([Block(dim, num_heads) for _ in range(depth)])
        self.ln = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab_size, bias=False)
        self.apply(lambda m: nn.init.normal_(m.weight, std=0.02) if isinstance(m, (nn.Linear, nn.Embedding)) else None)

    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for block in self.blocks:
            x = block(x)
        return self.head(self.ln(x))

count = lambda m, trainable=False: sum(p.numel() for p in m.parameters() if p.requires_grad or not trainable)

def loss_fn(model, x, y):
    return F.cross_entropy(model(x).reshape(-1, vocab_size), y.reshape(-1))

@torch.no_grad()
def estimate_loss(model, task, split="val", iters=40):
    model.eval()
    out = sum(loss_fn(model, *get_batch(task, split)).item() for _ in range(iters)) / iters
    model.train()
    return out

def train(model, task, iters, lr, log=None):
    # optimizes only the parameters with requires_grad = True -> [(iter, val loss)]
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    curve = []
    model.train()
    for it in range(iters + 1):
        if it % max(1, iters // 5) == 0:
            curve.append((it, estimate_loss(model, task)))
            if log:
                print(f"[{log}] iter {it:5d} | val loss {curve[-1][1]:.4f}")
        if it == iters:
            break
        loss = loss_fn(model, *get_batch(task, "train"))
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()
    return curve

## 3. Pretrain on Shakespeare
This plays the role of the large pretrained model. After this cell its weights are never trained directly again, except in the full fine-tuning baseline.

In [ ]:
pretrain_iters = 3000

base = GPT(vocab_size, block_size).to(device)
print(f"base params: {count(base) / 1e6:.2f}M")
train(base, "shakespeare", pretrain_iters, lr=1e-3, log="pretrain")
zero_shot = estimate_loss(base, "names")
print(f"pretrained model on names, no adaptation: val loss {zero_shot:.4f}")

## 4. The LoRA layer
### Formula: Low-rank update
$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r}\, B A x$$
- $W_0 \in \mathbb{R}^{d_{out} \times d_{in}}$: pretrained weight, **frozen** (no gradient)
- $A \in \mathbb{R}^{r \times d_{in}}$, $B \in \mathbb{R}^{d_{out} \times r}$: the only trained parameters, $r \ll \min(d_{in}, d_{out})$
- $\Delta W = BA$ has rank at most $r$
- trainable parameters: $r\,(d_{in} + d_{out})$ instead of $d_{in} \cdot d_{out}$. For $d = 192$, $r = 8$: 3,072 instead of 36,864

### Formula: Initialization
$$A \sim \text{random (Kaiming uniform)}, \qquad B = 0 \quad \Rightarrow \quad \Delta W = BA = 0 \ \text{at the start}$$
- the adapted model starts out **exactly equal** to the pretrained one; training moves away from it smoothly
- (both zero would give zero gradients forever; both random would perturb the model at step 0)

### Formula: Scaling
$$\text{scale} = \frac{\alpha}{r}$$
- $\alpha$: a constant (often set to $r$ or $2r$ and then left alone); dividing by $r$ keeps the update's magnitude roughly stable when $r$ is changed, so the learning rate need not be retuned

### Formula: Merging for inference
$$W = W_0 + \frac{\alpha}{r} B A$$
- a single matrix of the original shape: **zero added latency**, unlike adapter layers
- subtracting $\frac{\alpha}{r} BA$ recovers $W_0$, so one base model can switch between many tasks

In [ ]:
class LoRALinear(nn.Module):
    # wraps a frozen nn.Linear: x (.., d_in) -> W0 x + (alpha / r) * B A x
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                                # W0 frozen
        self.r, self.scale = r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))    # (r, d_in)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # (d_out, r), zero -> delta W = 0 at init
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale   # (.., d_in) -> (.., r) -> (.., d_out)

    def delta(self):
        return self.scale * self.B @ self.A                        # (d_out, d_in), rank <= r

    @torch.no_grad()
    def merge(self):
        # -> a plain nn.Linear with W0 + delta W
        merged = nn.Linear(self.base.in_features, self.base.out_features, bias=self.base.bias is not None).to(self.A.device)
        merged.weight.copy_(self.base.weight + self.delta())
        if self.base.bias is not None:
            merged.bias.copy_(self.base.bias)
        return merged


def add_lora(model, r, alpha=None, targets=("q", "v")):
    # returns a copy of `model` with everything frozen and LoRA on the chosen attention matrices
    model = copy.deepcopy(model)
    for p in model.parameters():
        p.requires_grad_(False)
    for block in model.blocks:
        for name in targets:
            setattr(block.attn, name, LoRALinear(getattr(block.attn, name), r, alpha or 2 * r).to(device))
    return model


def merge_lora(model):
    model = copy.deepcopy(model)
    for block in model.blocks:
        for name in ("q", "k", "v", "o"):
            layer = getattr(block.attn, name)
            if isinstance(layer, LoRALinear):
                setattr(block.attn, name, layer.merge())
    return model

lora = add_lora(base, r=8)
print(f"total params {count(lora)} | trainable {count(lora, trainable=True)} ({100 * count(lora, trainable=True) / count(base):.2f}% of the base model)")
x, _ = get_batch("names", "val")
with torch.no_grad():
    print("B = 0 at init, so outputs equal the base model:", torch.allclose(lora(x), base(x)))

## 5. Adapt to names: full fine-tuning vs LoRA
Which matrices to adapt? The paper finds that, for a fixed parameter budget, putting LoRA on $W_q$ **and** $W_v$ beats spending the same budget on one matrix at a higher rank. That is the default here.

Three ways to adapt, same data and number of steps:
- **full fine-tuning**: every weight trains (the reference)
- **head only**: only the output layer trains (the cheapest classical baseline)
- **LoRA** $r = 8$ on $W_q, W_v$

LoRA uses a higher learning rate than full fine-tuning, as is usual: $B$ starts at zero and only a few parameters move.

In [ ]:
adapt_iters = 1500

full = copy.deepcopy(base)
curve_full = train(full, "names", adapt_iters, lr=3e-4, log="full fine-tune")

head_only = copy.deepcopy(base)
for name, p in head_only.named_parameters():
    p.requires_grad_(name.startswith("head"))
curve_head = train(head_only, "names", adapt_iters, lr=3e-3, log="head only")

lora = add_lora(base, r=8)
curve_lora = train(lora, "names", adapt_iters, lr=3e-3, log="LoRA r=8")

print(f"\n{'method':16s} {'trainable':>10s} {'names val':>10s} {'shakespeare val':>16s}")
print(f"{'no adaptation':16s} {0:10d} {zero_shot:10.4f} {estimate_loss(base, 'shakespeare'):16.4f}")
for name, m in [("full fine-tune", full), ("head only", head_only), ("LoRA r=8", lora)]:
    print(f"{name:16s} {count(m, trainable=True):10d} {estimate_loss(m, 'names'):10.4f} {estimate_loss(m, 'shakespeare'):16.4f}")

plt.plot(*zip(*curve_full), marker="o", label="full fine-tune")
plt.plot(*zip(*curve_head), marker="o", label="head only")
plt.plot(*zip(*curve_lora), marker="o", label="LoRA r=8 (q, v)")
plt.xlabel("iter"); plt.ylabel("names val loss"); plt.legend(); plt.show()

## 6. Merge, and check nothing changed
Fold $\frac{\alpha}{r} BA$ into the frozen weights. The merged model has exactly the base model's architecture and parameter count and must give the same outputs as the LoRA model.

In [ ]:
merged = merge_lora(lora)
with torch.no_grad():
    diff = (merged(x) - lora(x)).abs().max().item()
print(f"params: base {count(base)} | LoRA model {count(lora)} | merged {count(merged)}")
print("max |merged - LoRA| on logits:", diff)

@torch.no_grad()
def generate(model, start="\n", n=120, temperature=0.8):
    model.eval()
    idx = torch.tensor([encode(start)], device=device)
    for _ in range(n):
        probs = F.softmax(model(idx[:, -block_size:])[:, -1] / temperature, dim=-1)
        idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
    return decode(idx[0].tolist())

print("--- base ---\n" + generate(base))
print("--- merged LoRA ---\n" + generate(merged))

## 7. How much rank is needed?
Sweep $r$. The paper's surprising finding is that very small ranks (even $r = 1$ or 2) already recover most of the gain on many tasks.

Then look at the learned update itself: the singular values of $\Delta W = \frac{\alpha}{r} BA$ for one layer. By construction at most $r$ of them are non-zero; how fast they decay shows how many directions the adaptation actually used.

In [ ]:
ranks = [1, 2, 4, 8, 16]
results = {}
for r in ranks:
    torch.manual_seed(0)
    m = add_lora(base, r=r)
    train(m, "names", adapt_iters, lr=3e-3)
    results[r] = (count(m, trainable=True), estimate_loss(m, "names"))
    print(f"r = {r:2d} | trainable {results[r][0]:6d} | names val loss {results[r][1]:.4f}")

full_loss = estimate_loss(full, "names")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(ranks, [results[r][1] for r in ranks], marker="o", label="LoRA")
ax[0].axhline(full_loss, ls="--", c="gray", label="full fine-tune"); ax[0].axhline(zero_shot, ls=":", c="C3", label="no adaptation")
ax[0].set_xscale("log", base=2); ax[0].set_xlabel("rank r"); ax[0].set_ylabel("names val loss"); ax[0].legend()

for i, block in enumerate(lora.blocks):
    s = torch.linalg.svdvals(block.attn.q.delta().detach())      # (D,) singular values of delta W_q
    ax[1].semilogy(range(1, 17), s[:16].cpu(), marker=".", label=f"layer {i}")
ax[1].set_xlabel("singular value index"); ax[1].set_ylabel("singular value of $\\Delta W_q$ (r = 8)"); ax[1].legend()
plt.show()

## Notes
- What LoRA saves is **training** memory and storage, not inference compute: no gradients or optimizer state for the frozen weights, and a checkpoint of a few MB per task instead of a full copy. The paper reports 10,000x fewer trainable parameters and 3x less GPU memory on GPT-3 with no loss in quality.
- Because $W_0$ is untouched, adapters are composable and removable: serve one base model with many LoRAs (swap per request), or keep it un-merged to add and subtract tasks. Image-generation "style LoRAs" are this same mechanism applied to a diffusion model's attention layers (`diffusion.ipynb`, `dit.ipynb`).
- Full fine-tuning overwrites the base behavior (check the Shakespeare column in §5); LoRA can only move within a rank-$r$ subspace per matrix, which also limits how much it forgets.
- Choices that matter in practice: which matrices (attention $q, v$ in the paper; all linear layers including the MLP in most current recipes), $r$ (4-64), $\alpha$ (commonly $2r$), and a higher learning rate than full fine-tuning.
- **QLoRA** (Dettmers et al., 2023): quantize the frozen base to 4 bits and train LoRA adapters in full precision on top, which makes fine-tuning a 7B-70B model possible on a single GPU. **DoRA** splits $W$ into magnitude and direction and applies LoRA to the direction. **rsLoRA** scales by $\alpha / \sqrt{r}$ for stability at high ranks.
- LoRA is the standard tool for the supervised fine-tuning and preference-optimization steps listed in `models.md` when the base model is too large to update in full.